<H3>PRI 2025/26: first project delivery</H3>

**GROUP X**
- name, number
- name, number
- name, number

<H3>Part 0: data loading</H3>

In [9]:

import os
from pathlib import Path

def load_bbc_dataset(root_dir):
    root = Path(root_dir)
    docs = []
    refs = []
    for category in ["business", "entertainment", "politics", "sport", "tech"]:
        art_dir = root / "News Articles" / category
        sum_dir = root / "Summaries" / category
        for art_file in sorted(art_dir.glob("*.txt")):
            sum_file = sum_dir / art_file.name
            if not sum_file.exists():
                continue
            # more robust decoding
            with art_file.open(encoding="utf-8", errors="replace") as f:
                d = f.read().strip()
            with sum_file.open(encoding="utf-8", errors="replace") as f:
                r = f.read().strip()
            docs.append({"id": art_file.stem, "category": category, "text": d})
            refs.append({"id": art_file.stem, "category": category, "summary": r})
    return docs, refs
   
## test 
docs, refs = load_bbc_dataset("data/BBC News Summary")
print(len(docs), len(refs))
print(docs[0])
print(refs[0])




2225 2225
{'id': '001', 'category': 'business', 'text': 'Ad sales boost Time Warner profit\n\nQuarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.\n\nThe firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.\n\nTime Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL\'s underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service f

<H3>Part I: demo of facilities</H3>

A) **Indexing** (preprocessing and indexing options)

In [10]:
import time
import sys
import spacy
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    doc = nlp(text)

    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

    noun_phrases = [
        chunk.text.lower().replace(" ", "_")
        for chunk in doc.noun_chunks
    ]

    return tokens + noun_phrases


def indexing(D, args=None):

    start = time.time()

    processed_docs = {}
    inverted_index = defaultdict(list)

    texts_for_tfidf = []
    doc_ids = []

    for doc in D:

        doc_id = doc["id"]
        tokens = preprocess_text(doc["text"])

        processed_docs[doc_id] = tokens
        doc_ids.append(doc_id)

        texts_for_tfidf.append(" ".join(tokens))

        # build inverted index
        term_counts = {}
        for t in tokens:
            term_counts[t] = term_counts.get(t, 0) + 1

        for term, tf in term_counts.items():
            inverted_index[term].append((doc_id, tf))

    # TF-IDF lexical vectors
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts_for_tfidf)

    # dense semantic index
    texts = [doc["text"] for doc in D]
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = encoder.encode(texts)

    I = {
        "doc_ids": doc_ids,
        "inverted_index": inverted_index,
        "tfidf_vectorizer": vectorizer,
        "tfidf_matrix": tfidf_matrix,
        "encoder": encoder,
        "embeddings": embeddings
    }

    indexing_time = time.time() - start

    memory_size = (
        sys.getsizeof(I)
        + embeddings.nbytes
    )

    return I, indexing_time, memory_size

In [11]:
I, build_time, mem_bytes = indexing(docs, args={"use_dense": True})
print("Index built in", build_time, "seconds")
print("Approximate index size:", mem_bytes / (1024**2), "MB")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built in 147.48664903640747 seconds
Approximate index size: 3.2595367431640625 MB


B) **Extractive summarization**

*B.1 Summarization baseline solution: results for a given document*

In [12]:
#code, statistics and/or charts here
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import numpy as np
import spacy

nlp = spacy.load("en_core_web_sm")


def mmr_rerank(sentences, scores, vectors, p, lambda_param=0.7):

    selected = []
    candidates = list(range(len(sentences)))

    while len(selected) < p and candidates:

        mmr_scores = []

        for i in candidates:

            relevance = scores[i]

            redundancy = 0
            if selected:
                redundancy = max(
                    cosine_similarity(
                        vectors[i].reshape(1,-1),
                        vectors[j].reshape(1,-1)
                    )[0][0]
                    for j in selected
                )

            mmr_score = lambda_param * relevance - (1-lambda_param) * redundancy

            mmr_scores.append((i, mmr_score))

        best = max(mmr_scores, key=lambda x: x[1])[0]

        selected.append(best)
        candidates.remove(best)

    return selected


def summarization(d, p=3, l=None, I=None, args=None):

    if args is None:
        args = {"method": "tfidf"}
    
    use_mmr = args.get("use_mmr", False)
    lambda_param = args.get("lambda_param", 0.7)

    doc = nlp(d)

    sentences = [sent.text.strip() for sent in doc.sents]

    # limit number of sentences if document is very small
    if p > len(sentences):
        p = len(sentences)

    method = args.get("method", "tfidf")

    # ---------- TF-IDF summarization ----------
    if method == "tfidf":

        vectorizer = I["tfidf_vectorizer"]

        sent_vectors = vectorizer.transform(sentences)
        doc_vector = vectorizer.transform([d])

        scores = cosine_similarity(sent_vectors, doc_vector).flatten()
        vectors_for_mmr = sent_vectors 

    # ---------- embedding summarization ----------
    elif method == "embedding":

        encoder = I["encoder"]

        sent_embeddings = encoder.encode(sentences)
        doc_embedding = encoder.encode([d])

        scores = cosine_similarity(sent_embeddings, doc_embedding).flatten()
        vectors_for_mmr = sent_embeddings

        
    elif method == "bm25":

        tokenized_sentences = [s.split() for s in sentences]
        bm25 = BM25Okapi(tokenized_sentences)

        query = d.split()

        scores = bm25.get_scores(query)

        vectorizer = I["tfidf_vectorizer"]
        vectors_for_mmr = vectorizer.transform(sentences)

    else:
        raise ValueError("Unknown summarization method")

    # rank sentences
    if use_mmr:
        # Convert sparse matrix to dense if needed for cosine_similarity
        if hasattr(vectors_for_mmr, "toarray"):
            vecs = vectors_for_mmr.toarray()
        else:
            vecs = vectors_for_mmr

        selected_indices = mmr_rerank(
            sentences,
            scores,
            vecs,
            p,
            lambda_param=lambda_param
        )
        result = [(int(i), float(scores[i])) for i in selected_indices]
    else:
        ranked = np.argsort(scores)[::-1]
        result = [(int(i), float(scores[i])) for i in ranked[:p]]

    return result

doc = docs[0]["text"]

summary = summarization(doc, p=3, I=I, args={"method":"tfidf"})

print(summary)

sentences = [s.text for s in nlp(doc).sents]

for i,score in summary:
    print(sentences[i], score)


ModuleNotFoundError: No module named 'rank_bm25'

*B.2 IR models (TF-IDF, BM25, lightweight encoder LLMs)*

In [24]:
#code, statistics and/or charts here
for method in ["tfidf","bm25","embedding"]:

    print("\nMODEL:",method)

    summary = summarization(
        doc,
        p=3,
        I=I,
        args={"method":method}
    )

    for i,score in summary:
        print(sentences[i])


MODEL: tfidf
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.
But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.

MODEL: bm25
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
But its film

*B.4 Maximal Marginal Relevance*

In [30]:
#code, statistics and/or charts here
    
methods = [
    ("tfidf", "TF-IDF + MMR"),
    ("embedding", "Embeddings (MiniLM) + MMR"),
    ("bm25", "BM25 + MMR"),
]

for method_key, method_label in methods:
    print(f"Method: {method_label} (λ = 0.5)")
    print("-" * 80)

    summary = summarization(
        doc,
        p=3,
        I=I,
        args={"method": method_key, "use_mmr": True, "lambda_param": 0.5}
    )

    for idx, score in summary:
        # print sentence index, score, and text
        print(f"[{idx:02d}] score = {score:.4f}")
        print(sentences[idx])
        print()

    print()  # extra blank line between methods


Method: TF-IDF + MMR (λ = 0.5)
--------------------------------------------------------------------------------
[17] score = 0.6800
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.

[11] score = 0.6220
But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.

[12] score = 0.6307
For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.


Method: Embeddings (MiniLM) + MMR (λ = 0.5)
--------------------------------------------------------------------------------
[00] score = 0.8281
Ad sales boost Time Warn

C) **Abstractive summarization**

*C.1 Prompting decoder LLMs*

In [28]:
#code, statistics and/or charts here
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# small decoder model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def rank_tfidf(query, I, top_k=3):
    """
    Return top_k most relevant documents using TF-IDF.
    Output: [(doc_id, score), ...]
    """
    vectorizer = I["tfidf_vectorizer"]
    tfidf_matrix = I["tfidf_matrix"]
    doc_ids = I["doc_ids"]

    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]

    ranked_idx = np.argsort(sims)[::-1][:top_k]
    return [(doc_ids[i], float(sims[i])) for i in ranked_idx]


def prompting_decoder_llm(query, I, D, top_k=3, max_new_tokens=100):
    """
    1. retrieve top documents with TF-IDF
    2. build prompt from their text
    3. generate abstractive summary
    """
    # get top documents
    ranked_docs = rank_tfidf(query, I, top_k=top_k)

    # make id -> document dictionary
    doc_lookup = {doc["id"]: doc for doc in D}

    # collect text of top docs
    context_parts = []
    for doc_id, score in ranked_docs:
        context_parts.append(doc_lookup[doc_id]["text"])

    context = "\n\n".join(context_parts)

    # build prompt
    prompt = (
        f"Summarize the following information about: {query}\n\n"
        f"{context[:2000]}\n\n"
        f"Summary:"
    )

    # tokenize prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # generate text
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # keep only generated summary part
    if "Summary:" in full_text:
        summary = full_text.split("Summary:", 1)[-1].strip()
    else:
        summary = full_text.strip()

    return {
        "query": query,
        "ranked_docs": ranked_docs,
        "prompt": prompt,
        "summary": summary
    }

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [29]:
query = "UK election campaign"

result = prompting_decoder_llm(
    query=query,
    I=I,
    D=docs,
    top_k=3,
    max_new_tokens=80
)

print("QUERY:")
print(result["query"])

print("\nTOP DOCUMENTS:")
for doc_id, score in result["ranked_docs"]:
    print(doc_id, round(score, 4))

print("\nPROMPT PREVIEW:")
print(result["prompt"][:800])

print("\nGENERATED SUMMARY:")
print(result["summary"])

QUERY:
UK election campaign

TOP DOCUMENTS:
415 0.2674
207 0.2592
249 0.2304

PROMPT PREVIEW:
Summarize the following information about: UK election campaign

Wilkinson to miss Ireland match

England will have to take on Ireland in the Six Nations without captain and goal-kicker Jonny Wilkinson, according to his Newcastle boss Rob Andrew.

Wilkinson - who had targeted the 27 February match for his international comeback - has been missed by England, not least for his goal-kicking. "Jonny's not fit yet," Falcons chief Andrew told BBC Radio Five Live. "He won't be fit for Dublin, there's no doubt about that, but he might be fit for Scotland and Italy." The 25-year-old has not played for England since the 2003 World Cup final after a succession of injuries. England, who have lost three Six Nations games in a row, wasted a 17-6 half-time lead in their 18-17 defeat to France. Goal-kicke

GENERATED SUMMARY:
England have a lot of talent to play in the Premier League and they have a good chanc

*C.2 Reciprocal rank funsion*

In [30]:
#code, statistics and/or charts here
from collections import defaultdict


def rank_dense(query, I, top_k=3):
    """
    Return top_k most relevant documents using dense embeddings.
    Output: [(doc_id, score), ...]
    """
    encoder = I["encoder"]
    embeddings = I["embeddings"]
    doc_ids = I["doc_ids"]

    query_emb = encoder.encode([query])
    sims = cosine_similarity(query_emb, embeddings)[0]

    ranked_idx = np.argsort(sims)[::-1][:top_k]
    return [(doc_ids[i], float(sims[i])) for i in ranked_idx]


def reciprocal_rank_fusion(rankings, k=60):
    """
    Combine multiple rankings using Reciprocal Rank Fusion.
    rankings = [ranking1, ranking2, ...]
    each ranking = [(doc_id, score), ...]
    """
    fused_scores = defaultdict(float)

    for ranking in rankings:
        for rank, (doc_id, score) in enumerate(ranking, start=1):
            fused_scores[doc_id] += 1.0 / (k + rank)

    fused_rank = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return fused_rank


def prompting_decoder_llm_rrf(query, I, D, top_k=3, max_new_tokens=100):
    """
    1. retrieve top documents with TF-IDF
    2. retrieve top documents with dense embeddings
    3. combine rankings with RRF
    4. build prompt from fused top documents
    5. generate abstractive summary
    """
    tfidf_rank = rank_tfidf(query, I, top_k=top_k)
    dense_rank = rank_dense(query, I, top_k=top_k)
    ranked_docs = reciprocal_rank_fusion([tfidf_rank, dense_rank], k=60)[:top_k]

    doc_lookup = {doc["id"]: doc for doc in D}

    context_parts = []
    for doc_id, score in ranked_docs:
        context_parts.append(doc_lookup[doc_id]["text"])

    context = "\n\n".join(context_parts)

    prompt = (
        f"Summarize the following information about: {query}\n\n"
        f"{context[:2000]}\n\n"
        f"Summary:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Summary:" in full_text:
        summary = full_text.split("Summary:", 1)[-1].strip()
    else:
        summary = full_text.strip()

    return {
        "query": query,
        "tfidf_rank": tfidf_rank,
        "dense_rank": dense_rank,
        "ranked_docs": ranked_docs,
        "prompt": prompt,
        "summary": summary
    }

In [31]:
query = "UK election campaign"

result = prompting_decoder_llm_rrf(
    query=query,
    I=I,
    D=docs,
    top_k=3,
    max_new_tokens=80
)

print("QUERY:")
print(result["query"])

print("\nTF-IDF RANK:")
for doc_id, score in result["tfidf_rank"]:
    print(doc_id, round(score, 4))

print("\nDENSE RANK:")
for doc_id, score in result["dense_rank"]:
    print(doc_id, round(score, 4))

print("\nFUSED RANK:")
for doc_id, score in result["ranked_docs"]:
    print(doc_id, round(score, 4))

print("\nPROMPT PREVIEW:")
print(result["prompt"][:800])

print("\nGENERATED SUMMARY:")
print(result["summary"])

QUERY:
UK election campaign

TF-IDF RANK:
415 0.2674
207 0.2592
249 0.2304

DENSE RANK:
415 0.5821
259 0.568
261 0.5607

FUSED RANK:
415 0.0328
207 0.0161
259 0.0161

PROMPT PREVIEW:
Summarize the following information about: UK election campaign

Wilkinson to miss Ireland match

England will have to take on Ireland in the Six Nations without captain and goal-kicker Jonny Wilkinson, according to his Newcastle boss Rob Andrew.

Wilkinson - who had targeted the 27 February match for his international comeback - has been missed by England, not least for his goal-kicking. "Jonny's not fit yet," Falcons chief Andrew told BBC Radio Five Live. "He won't be fit for Dublin, there's no doubt about that, but he might be fit for Scotland and Italy." The 25-year-old has not played for England since the 2003 World Cup final after a succession of injuries. England, who have lost three Six Nations games in a row, wasted a 17-6 half-time lead in their 18-17 defeat to France. Goal-kicke

GENERATED SUMMA

In [ ]:
queries = [
    "rugby player injury",
    "football club transfer deal",
    "technology company earnings",
    "prime minister policy speech",
    "music award nominations"
]

for q in queries:
    print("\n", "="*80)
    print("QUERY:", q)

    r1 = prompting_decoder_llm(q, I, docs, top_k=5)
    r2 = prompting_decoder_llm_rrf(q, I, docs, top_k=5)

    print("C1 docs:", r1["ranked_docs"])
    print("C2 docs:", r2["ranked_docs"])

C1 SUMMARY:
The technology is a combination of a combination with a combination that can be used to monitor computer activities, and a combination which can be combined with a combined that can also be combined to monitor the computer activity.
The company has been working on a system that can monitor computers for a number of years.
It is not clear how the system will be used, but it is expected to be used


C2 SUMMARY:
The technology is a combination of a combination with a combination that can be used to monitor computer activities, and a combination which can be combined with a combined that can also be combined to monitor the computer activity.
The company has been working on a system that can monitor computers for a number of years.
It is not clear how the system will be used, but it is expected to be used


D) **Keyword extraction**

In [ ]:
#code, statistics and/or charts here

E) **Evaluation**

In [ ]:
#code, statistics and/or charts here

<H3>Part II: questions materials (optional)</H3>

**(1)** Keyword extraction using classic *vs* generative stances

In [ ]:
#code, statistics and/or charts here

**(2)** Summarization performance for the overall and category-conditional corpora.

In [ ]:
#code, statistics and/or charts here

**...** (additional questions with empirical results)

<H3>END</H3>